In [1]:
#load all five datasets without changing the original files
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.io import arff
from sklearn.model_selection import train_test_split

with open("attributenames.txt") as file:
    names = file.readlines()
names = [s.strip()[s.find("\t") + 1:] for s in names]

datasets = {}
for year in range(1, 6):
    data = pd.DataFrame(arff.loadarff(f"data/raw/{year}year.arff")[0])
    data.columns = names
    data.rename(columns={"": "bankrupt"}, inplace=True)
    data["bankrupt"] = data["bankrupt"].astype(int)
    datasets[year] = data

In [ ]:
#compare all five file, UCI file numbers are not years ahead
#Ex: 1year.arff predicts 5 years ahead, 5year.arff predicts 1 year ahead
summary_rows = []
for year, data in datasets.items():
    summary_rows.append({
        "File": f"{year}year.arff",
        "Years Ahead": 6 - year,
        "Rows": len(data),
        "Bankrupt": int(data["bankrupt"].sum()),
        "Bankrupt Percent": data["bankrupt"].mean() * 100,
        "Rows With Missing Values": int(data.isna().any(axis=1).sum()),
        "Missing Row Percent": data.isna().any(axis=1).mean() * 100,
        "Duplicate Rows": int(data.duplicated().sum())
    })
dataset_summary = pd.DataFrame(summary_rows).set_index("File")
dataset_summary.round(2)

,Years Ahead,Rows,Bankrupt,Bankrupt Percent,Rows With Missing Values,Missing Row Percent,Duplicate Rows
File,,,,,,,
1year.arff,5,7027,271,3.86,3833,54.55,82
2year.arff,4,10173,400,3.93,6085,59.82,90
3year.arff,3,10503,495,4.71,5618,53.49,87
4year.arff,2,9792,515,5.26,5023,51.30,82
5year.arff,1,5910,410,6.94,2879,48.71,60


In [ ]:
#remove identical full rows, keeping the first copy
#We are keeping extreme values, there is no evidence to suggest they are faulty. 
unique_datasets = {}
for year, data in datasets.items():
    unique = data.drop_duplicates().copy()
    features = unique.drop(columns="bankrupt")
    assert unique["bankrupt"].isin([0, 1]).all()
    assert not unique["bankrupt"].isna().any()
    assert not features.isin([np.inf, -np.inf]).any().any()
    #stop if identical features have different labels 
    assert not features.duplicated().any()
    unique_datasets[year] = unique

In [4]:
#split each year's data into 70% training, 15% validation and 15% test
#stratify keeps similar bankruptcy proportions

splits = {}
for year, data in unique_datasets.items():
    train, remaining = train_test_split(
        data, test_size=0.30, stratify=data["bankrupt"], random_state=42)
    validation, test = train_test_split(
        remaining, test_size=0.50, stratify=remaining["bankrupt"], random_state=42)
    assert not set(train.index) & set(validation.index)
    assert not set(train.index) & set(test.index)
    assert not set(validation.index) & set(test.index)
    assert len(train) + len(validation) + len(test) == len(data)
    splits[year] = {"train": train, "validation": validation, "test": test}

In [ ]:
#Fill missing features using training medians only
#Use the same medians for validation and test

#why median? 
# deleting incomplete rows in 1year.arff loses 241 of 271 bankrupt cases
# the median is less affected by extreme ratios (which we have) than the mean
# training medians avoid using validation/test information during preprocessing

cleaned_splits = {}
training_medians = {}
cleaning_rows = []
for year, year_splits in splits.items():
    medians = year_splits["train"].drop(columns="bankrupt").median()
    assert not medians.isna().any(), "A training feature has no observed values"
    training_medians[year] = medians
    cleaned_splits[year] = {}
    for split_name, data in year_splits.items():
        filled = data.fillna(medians)
        assert not filled.isna().any().any()
        assert len(filled) == len(data)
        assert filled["bankrupt"].equals(data["bankrupt"])
        #check that every observed feature value stayed unchanged
        observed = data.drop(columns="bankrupt").notna()
        assert filled.drop(columns="bankrupt").where(observed).equals(
            data.drop(columns="bankrupt").where(observed))
        cleaned_splits[year][split_name] = filled
        cleaning_rows.append({"File": f"{year}year.arff", "Years Ahead": 6 - year, "Split": split_name,
            "Rows": len(filled), "Bankrupt": int(filled["bankrupt"].sum()),
            "Values Filled": int(data.isna().sum().sum()),
            "Missing Values Remaining": int(filled.isna().sum().sum())})
cleaning_summary = pd.DataFrame(cleaning_rows)
cleaning_summary

,File,Years Ahead,Split,Rows,Bankrupt,Values Filled,Missing Values Remaining
0,1year.arff,5,train,4861,190,4098,0
1,1year.arff,5,validation,1042,41,851,0
2,1year.arff,5,test,1042,40,821,0
3,2year.arff,4,train,7058,279,8241,0
4,2year.arff,4,validation,1512,59,1926,0
5,2year.arff,4,test,1513,60,1918,0
6,3year.arff,3,train,7291,345,7017,0
7,3year.arff,3,validation,1562,74,1437,0
8,3year.arff,3,test,1563,74,1379,0
9,4year.arff,2,train,6797,359,6146,0


In [ ]:
#save three cleaned datasets per file; keep checking records in their own folder
output_folder = Path("data/cleaned")
records_folder = output_folder / "supporting_records"
records_folder.mkdir(parents=True, exist_ok=True)
for year, year_splits in cleaned_splits.items():
    year_folder = output_folder / f"{year}year"
    year_folder.mkdir(parents=True, exist_ok=True)
    for split_name, data in year_splits.items():
        data.to_csv(year_folder / f"{year}year_{split_name}.csv", index=False)
        pd.Series(data.index, name="Original Row Index").to_csv(
            records_folder / f"{year}year_{split_name}_row_ids.csv", index=False)
    training_medians[year].rename("Training Median").to_csv(
        records_folder / f"{year}year_training_medians.csv")
dataset_summary.to_csv(records_folder / "dataset_summary.csv")
cleaning_summary.to_csv(records_folder / "cleaning_summary.csv", index=False)

In [ ]:
'''
CLEANING CONCLUSIONS:
- Duplicate rows were removed
- missing features were filled with medians calculated from each year's training set
- extreme values were kept, because we have no information to say they are not valid (Small denominators one possibility)
'''

"\nCLEANING CONCLUSIONS:\n\n- exact duplicate rows were removed; the first copy was kept\n- no rows were removed just because they contained missing values\n- missing features were filled with medians calculated from each year's training set\n- extreme values were kept; scaling and class balancing have not been applied\n- the five datasets stay separate; this random split does not establish future-time or new-company performance\n- no models were trained or evaluated here\n"